In [1]:
import sys
import os

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
from utils.ingest import load_faq_data

In [4]:
documents = load_faq_data()

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

In [9]:
class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from anthropic import Anthropic

In [12]:
load_dotenv()

True

In [13]:
anthropic_client = Anthropic()

In [14]:
import json

In [15]:
user_prompt = json.dumps(doc)

In [16]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [17]:
messages = [
    {'role': 'user', 'content': user_prompt}
]

In [18]:
response = anthropic_client.messages.parse(
    model="claude-haiku-4-5",
    system=data_gen_instructions,
    max_tokens=1024,
    messages=messages,
    output_format=Questions
)

In [19]:
result = response.parsed_output

In [20]:
result.questions

['Is it too late to sign up for this course?',
 'Do I need to join before a certain date to get a certificate?',
 'What happens if I start the course late - can I still get certified?',
 'Is there a deadline for submitting the final project to earn the certificate?',
 'Can I join the course anytime or is there a cutoff date?']

In [21]:
response.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=354, output_tokens=86, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [22]:
from utils.evaluation_utils import llm_structured

In [23]:
result, usage = llm_structured(
    anthropic_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [24]:
print(result.questions)

['Is it too late to enroll in this course?', "What's the deadline for submitting my final project to get the certificate?", "Can I join the course now even though it's already started?", 'Do I need to complete everything before a certain date if I want to earn the certificate?', 'If I sign up late, will I still be able to get a certificate?']


In [25]:
usage.input_tokens, usage.output_tokens

(354, 89)

In [26]:
from utils.evaluation_utils import calc_price

In [27]:
cost = calc_price(usage)

In [28]:
cost

{'input_cost': 0.000354, 'output_cost': 0.000445, 'total_cost': 0.000799}

In [29]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to enroll in this course?',
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting my final project to get the certificate?",
  'document': '74eb249bbf'},
 {'question': "Can I join the course now even though it's already started?",
  'document': '74eb249bbf'},
 {'question': 'Do I need to complete everything before a certain date if I want to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up late, will I still be able to get a certificate?',
  'document': '74eb249bbf'}]

In [30]:
import json
from utils.evaluation_utils import llm_structured_retry

In [31]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        anthropic_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })
    return results, usage

In [32]:
from tqdm.auto import tqdm

In [31]:
ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.append(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

## Parallel processing

In [32]:
from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

In [34]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/85 [00:00<?, ?it/s]

In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

425

In [39]:
from utils.evaluation_utils import calc_total_price

In [40]:
calc_total_price(usages)

0.08450999999999999

In [41]:
import pandas as pd

In [42]:
df_ground_truth = pd.DataFrame(ground_truth)

In [47]:
df_ground_truth.head()

,question,document
0,Is it too late to start this course if I just ...,74eb249bbf
1,Do I need to join before a certain date to get...,74eb249bbf
2,Can I still enroll and complete the course if ...,74eb249bbf
3,What's the deadline for submitting my project ...,74eb249bbf
4,"If I start now, will I be able to get certifie...",74eb249bbf


In [44]:
df_ground_truth.shape

(425, 2)

In [46]:
df_ground_truth['document'].nunique()

85

In [48]:
df_ground_truth.to_csv("./data/ground_truth-new.csv", index=False)

In [39]:
df_ground_truth = df_ground_truth.to_dict("orient"="record")
sample = df_ground_truth[0]

KeyError: 0

In [56]:
print(sample['document'])
print(sample['question']) # Generated question

74eb249bbf
Is it too late to start this course if I just found out about it?


In [55]:
[doc for doc in documents if doc['id'] == sample['document']] # Original document

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}]

In [33]:
from utils.ingest import build_index

In [34]:
index = build_index(documents)

In [62]:
results_index_from_sample_question = index.search(
    sample['question'],
    num_results=5
)

In [65]:
[1 if doc['id'] == sample['document'] else 0 for doc in results_index_from_sample_question]

[0, 1, 0, 0, 0]

## Apply across the board to all documents

In [35]:
import pandas as pd

from embed.embedder import Embedder

## Load the ground truth

In [36]:
ground_truth = pd.read_csv('./data/ground_truth-new.csv')

In [37]:
ground_truth.head()

,question,document
0,Is it too late to start this course if I just ...,74eb249bbf
1,Do I need to join before a certain date to get...,74eb249bbf
2,Can I still enroll and complete the course if ...,74eb249bbf
3,What's the deadline for submitting my project ...,74eb249bbf
4,"If I start now, will I be able to get certifie...",74eb249bbf


In [38]:
ground_truth.shape

(425, 2)

In [39]:
ground_truth = ground_truth.to_dict(orient="records")

In [40]:
ground_truth[:5]

[{'question': 'Is it too late to start this course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to join before a certain date to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "Can I still enroll and complete the course if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting my project to earn the certificate?",
  'document': '74eb249bbf'},
 {'question': 'If I start now, will I be able to get certified for this course?',
  'document': '74eb249bbf'}]

## Load the model

In [41]:
model = Embedder("../embed/models/Xenova/all-MiniLM-L6-v2")

../embed/models/Xenova/all-MiniLM-L6-v2


## Text and Vector search functions 

In [42]:
def text_search (query, num_results=5):
    boost_dict = {"question": 3.0, "section": 0.5}
    
    return index.search(
        query,
        boost_dict=boost_dict,
        num_results=num_results
    )

In [61]:
def vector_search(query, num_results=5):
    #boost_dict = {"question": 3.0, "section": 0.5}
    query_vector = model.encode(query)
    
    return vindex.search(
        query_vector,
        #boost_dict=boost_dict,
        num_results=num_results
    )

## Compute relevance matrix for text search

In [44]:
def compute_relevance_text(q):
    doc_id = q['document'] # ID
    results = text_search(q['question'])

    return [1 if doc['id'] == doc_id else 0 for doc in results] 
    

In [45]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [47]:
relevance_total_text

[[0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

## Compute relevance matrix for any search function

In [48]:
def compute_relevance(q, search_function):
    doc_id = q['document']
    results = search_function(query=q['question'])

    return [1 if doc['id'] == doc_id else 0 for doc in results]

In [49]:
def compute_relevance_total(gr_tr, search_function):
    relevance_total = []
    for q in tqdm(gr_tr):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

### For text search

In [50]:
relevance_text_matrix = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/425 [00:00<?, ?it/s]

In [51]:
relevance_text_matrix

[[0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0,

## For Vector search

In [52]:
from minsearch import VectorSearch
from utils.ingest import build_vindex

In [53]:
texts = [doc['question'] + ' ' + doc['answer'] for doc in documents]

In [55]:
print(texts[:5])

['I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email? You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.", 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions? The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube

In [56]:
vectors = []
batch_size = 5

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_encoded = model.encode_batch(batch)
    vectors.extend(batch_encoded)

  0%|          | 0/17 [00:00<?, ?it/s]

In [59]:
vindex = build_vindex(documents, vectors)

In [62]:
relevance_vector_matrix = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/425 [00:00<?, ?it/s]

In [64]:
relevance_vector_matrix

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0,

# Metrics: Hit rate and Mean Reciprocal Rank (MRR)

## Compute the hit rate

In [65]:
def hit_rate(relevance_matrix):
    count = 0
    for d in relevance_matrix:
        if 1 in d:
            count += 1
    return count / len(relevance_matrix)

In [66]:
text_hit_rate = hit_rate(relevance_text_matrix)
print(text_hit_rate)

0.8423529411764706


## Compute Mean Reciprocal Rank

In [67]:
def mrr(relevance_matrix):
    total_score = 0
    
    for row in relevance_matrix:
        for rank in range(len(row)):
            if row[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_matrix)

In [68]:
text_mmr = mrr(relevance_text_matrix)
print(text_mmr)

0.693764705882353


In [69]:
def evaluate(gr_tr, search_function):
    relevance_total = compute_relevance_total(gr_tr, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [70]:
evaluate(ground_truth, text_search)

  0%|          | 0/425 [00:00<?, ?it/s]

{'hit_rate': 0.8423529411764706, 'mrr': 0.693764705882353}

In [71]:
evaluate(ground_truth, vector_search)

  0%|          | 0/425 [00:00<?, ?it/s]

{'hit_rate': 0.9670588235294117, 'mrr': 0.8856078431372547}

## Optimize the search function == research best parameters of boost_dict attribute

In [126]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [127]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/425 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8776470588235294, 'mrr': 0.7430588235294117}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8823529411764706, 'mrr': 0.7464313725490195}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.84, 'mrr': 0.6886666666666668}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8141176470588235, 'mrr': 0.6663137254901964}


  0%|          | 0/425 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7905882352941176, 'mrr': 0.6447450980392159}


In [128]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [129]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/425 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/425 [00:00<?, ?it/s]

In [130]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
4,1.0,2.0,0.2,0.924706,0.802275
20,2.0,4.0,0.5,0.927059,0.802078
3,1.0,2.0,0.1,0.920000,0.799961
35,5.0,10.0,0.5,0.920000,0.799961
19,2.0,4.0,0.2,0.920000,0.799961
7,1.0,4.0,0.2,0.910588,0.799686
18,2.0,4.0,0.1,0.922353,0.797373
8,1.0,4.0,0.5,0.903529,0.797294
34,5.0,10.0,0.2,0.922353,0.796980
6,1.0,4.0,0.1,0.915294,0.796392


## Evaluation with best params of text search

In [133]:
def text_search (query, num_results=5):
    boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.2}
    
    return index.search(
        query,
        boost_dict=boost_dict,
        num_results=num_results
    )

In [134]:
evaluate(ground_truth, text_search)

  0%|          | 0/425 [00:00<?, ?it/s]

{'hit_rate': 0.9247058823529412, 'mrr': 0.8022745098039213}

### Case with vector search function

In [84]:
from minsearch import VectorSearch

In [87]:
vindex = VectorSearch(
    keyword_fields=['course']
)

In [107]:
text = []
for doc in documents:
    text.append(doc['question'] + ' ' + doc['answer'])

In [108]:
text[0]

'I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [109]:
vectors = []
batch_size = 5

for i in tqdm(range(0, len(text), batch_size)):
    batch = text[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/17 [00:00<?, ?it/s]

TypeError: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]